# Chapter 6 &mdash; The Myhill&ndash;Nerode Theorem and DFA Isomorphism

**Concept 7 of the Chapter 6 decomposition:** *The Myhill–Nerode Theorem, and the Formal Definition of DFA Isomorphism*

Any two minimal DFA for the same regular language are isomorphic &mdash; uniqueness, spelled out.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Myhill-Nerode/Concept-Myhill-Nerode.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **Myhill&ndash;Nerode.** Any two **minimal** DFA for the same regular language are
> **isomorphic**.

An **isomorphism** is a bijection $f: Q_1 \to Q_2$ with

* $f(q_{0,1}) = q_{0,2}$,
* $q \in F_1 \iff f(q) \in F_2$,
* $f(\delta_1(q,a)) = \delta_2(f(q),a)$ for all $q, a$.

So a regular language has **one** minimal DFA, up to renaming. That is what makes
"build it twice and compare" (Chapter 5, Concept 4) a valid verification method, and
what gives the state count meaning as a property of the *language*.

## 2. Definitions

### Two very different-looking designs for one language

In [ ]:
A = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> IF
Od  : 0 -> IF
Od  : 1 -> Od
''')
# B tracks the pair (parity of 0s, parity of 1s) but is final on even-0s
# alone, so it recognises exactly A's language with twice as many states.
B = md2mc('''DFA
IF      : 0 -> Od0ev1
IF      : 1 -> Fev0od1
Fev0od1 : 0 -> Od0od1
Fev0od1 : 1 -> IF
Od0ev1  : 0 -> IF
Od0ev1  : 1 -> Od0od1
Od0od1  : 0 -> Fev0od1
Od0od1  : 1 -> Od0ev1
''')

### Find the isomorphism explicitly, by a paired BFS

In [ ]:
def find_iso(D1, D2):
    from collections import deque
    f = {D1["q0"]: D2["q0"]}; dq = deque([D1["q0"]])
    while dq:
        q = dq.popleft()
        if (q in D1["F"]) != (f[q] in D2["F"]): return None
        for a in sorted(D1["Sigma"]):
            s, t = step_dfa(D1, q, a), step_dfa(D2, f[q], a)
            if s in f:
                if f[s] != t: return None
            else:
                f[s] = t; dq.append(s)
    return f if len(set(f.values())) == len(f) else None

## 3. Tests

Both machines minimize to the same size &mdash; a necessary condition.

In [ ]:
mA, mB = min_dfa(A), min_dfa(B)
print("|Q| : A=%d B=%d   minimal: %d and %d"
      % (len(A["Q"]), len(B["Q"]), len(mA["Q"]), len(mB["Q"])))
print("same language? ", langeq_dfa(A, B))
assert langeq_dfa(A, B)
assert len(mA["Q"]) == len(mB["Q"])

And the isomorphism exists &mdash; here it is, state by state.

In [ ]:
f = find_iso(mA, mB)
print("isomorphism found:", f is not None)
assert f is not None
for k in sorted(f): print("   %-14s -> %s" % (k, f[k]))

Verify all three isomorphism conditions explicitly.

In [ ]:
assert f[mA["q0"]] == mB["q0"], "start states must correspond"
assert all((q in mA["F"]) == (f[q] in mB["F"]) for q in mA["Q"]), "finality preserved"
assert all(f[step_dfa(mA, q, a)] == step_dfa(mB, f[q], a)
           for q in mA["Q"] for a in mA["Sigma"]), "transitions preserved"
assert len(set(f.values())) == len(mA["Q"]), "bijection"
print("start, finality, transitions, bijectivity -- all four hold")
print("iso_dfa agrees :", iso_dfa(mA, mB))

Uniqueness in action: **any** correct design minimizes to the same machine.

In [ ]:
Cs = [A, B, union_dfa(A, A), intersect_dfa(A, A)]
sizes = [len(min_dfa(X)["Q"]) for X in Cs]
print("minimal sizes of four different constructions :", sizes)
assert len(set(sizes)) == 1
assert all(iso_dfa(min_dfa(X), mA) for X in Cs)
print("all isomorphic to the same minimal machine -- Myhill-Nerode.")

## 4. Exercises


1. Write down the isomorphism conditions as a commuting-diagram picture.
2. Why does the theorem need **minimal** DFA? Give a counterexample without it.
3. How does Myhill&ndash;Nerode justify "the number of states of $L$" as a well-defined quantity?

In [ ]:
# Your work for the exercises above.